# Stage 6 — table

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and an inline eval. The implementation itself is yours to write.


## 1. Setup


Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running anything else in the same runtime), run them top-to-bottom.


### Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### Install dependencies


Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### (no API key needed for this stage)


In [ ]:
# This stage doesn't call the OpenAI API.


### Seed prior stages' outputs from the reference run


Stage 6 reads outputs from earlier stages. Each Colab notebook gets its own runtime, so work done in another notebook is not visible here. This cell seeds `stage_01..stage_05/data/` from the canonical reference run so Stage 6 has inputs to work with — but only when the dir is empty, so re-running an earlier stage IN THIS runtime is not clobbered.


In [ ]:
# Load prior stages' reference outputs as inputs for Stage 6.
# Each Colab notebook opens with a fresh runtime, so any work done in a
# Stage <6 notebook in a DIFFERENT runtime is not visible here.
# This cell makes the stage runnable in isolation against the canonical
# reference run. If you re-run an earlier stage IN THIS runtime, your
# output replaces these reference files (cwd is /content/...).
import os, shutil, glob
for n in range(1, 6):
    dst = f"stage_0{n}/data"
    src = f"reference_outputs/stage_0{n}/data"
    if not os.path.isdir(src):
        continue
    os.makedirs(dst, exist_ok=True)
    # Only seed if the participant hasn't produced anything for this stage
    # in the current runtime — otherwise we'd clobber their work.
    if any(os.scandir(dst)):
        print(f"skip stage_0{n} — already has files (keeping your work)")
        continue
    for src_file in glob.glob(f"{src}/*"):
        shutil.copy(src_file, dst)
    print(f"seeded stage_0{n}/data from reference_outputs")


## 2. Spec — paste this into Gemini


Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
Flatten every `stage_05/data/*.json` into one row per
(paper × animal_arm) and write
`stage_06/data/mabs_animal_studies.csv` with these columns (in order):

  pmid, source_type, first_author, year, mab_name, target, format,
  development_stage, regulatory_context, threeRs_mentioned,
  author_reduction_recommendation, species, n_animals, study_type,
  duration_days, species_justification, cross_reactivity_evidence,
  endpoints_unique_to_animal, concurrent_nam, n_nams_discussed

`n_nams_discussed` is `len(rec["nams_discussed"])` — a count, not a list.

A paper with no animal_arms contributes zero rows. The header is always
written.

Skip any sibling JSON named `eval.json`, `eval_script.json`,
`eval_llm.json`, or `score.json`.
```


## 3. Gotchas Gemini probably won't know


Copy any that apply into Gemini if it goes off-track:

- **Use `csv.DictWriter(f, fieldnames=FIELDNAMES)`.** Keeps column
  order deterministic even when some arms are missing fields.
- **Open with `newline=""`** to avoid blank lines on Windows runtimes.
- **A paper-level `nams_discussed` list** needs `len(...)` not the
  list itself written to the cell.
- **Skip eval artifacts in the directory** (see spec).


## 4. Seed — a few lines to anchor Gemini


In [ ]:
import csv, glob, json, os
os.makedirs("stage_06/data", exist_ok=True)
SKIP_NAMES = {"eval.json", "eval_script.json", "eval_llm.json", "score.json"}
# Build a list `rows` of dicts (one per animal_arm), then write the CSV.


## 5. Your implementation


Drive Gemini to fill this in. Iterate until the inspect cell below shows reasonable output and the eval cell passes.


In [ ]:
# TODO: implement Stage 6 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the inspect + eval cells next.


## 6. Inspect output


In [ ]:
import csv
with open("stage_06/data/mabs_animal_studies.csv") as f:
    csv_rows = list(csv.DictReader(f))
print(f"{len(csv_rows)} animal-arm rows in stage_06/data/mabs_animal_studies.csv\n")
print("Sample (first 5 rows):")
for r in csv_rows[:5]:
    print(f"  PMID {r['pmid']:>10s} | {r['species']:>10s} | "
          f"{r['study_type']:>18s} | n={r.get('n_animals','?'):>4s} | "
          f"target={(r.get('target') or '')[:24]}")
n_nam = sum(1 for r in csv_rows if (r.get('concurrent_nam') or '').strip())
print(f"\n{n_nam}/{len(csv_rows)} rows have a concurrent_nam.")


## 7. Run eval


Inline eval — same checks as `eval/eval_06_script.py`, but the code is right here so you can see what it's measuring. Writes `stage_06/eval/eval_script.json` + `score.json`.


In [ ]:
# Same checks as eval/eval_06_script.py, inlined.
import csv, glob, json, os
from collections import Counter

os.makedirs("stage_06/eval", exist_ok=True)
REQUIRED_COLUMNS = {
    "pmid","species","study_type","year","mab_name","target","format",
    "development_stage","regulatory_context","threeRs_mentioned","n_animals",
    "duration_days","species_justification","cross_reactivity_evidence",
    "endpoints_unique_to_animal","concurrent_nam","n_nams_discussed",
    "author_reduction_recommendation","first_author",
}
SKIP = {"eval.json","eval_script.json","eval_llm.json","score.json"}

with open("stage_06/data/mabs_animal_studies.csv") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    columns = set(reader.fieldnames or [])

errors = []
for r in rows:
    if not r.get("pmid"):       errors.append("row missing pmid")
    if not r.get("species"):    errors.append(f"row {r.get('pmid')}: missing species")
    if not r.get("study_type"): errors.append(f"row {r.get('pmid')}: missing study_type")
    if r.get("n_animals"):
        try:
            if int(r["n_animals"]) < 0:
                errors.append(f"row {r.get('pmid')}: negative n_animals")
        except ValueError:
            errors.append(f"row {r.get('pmid')}: non-integer n_animals")

def _extracted_records():
    for p in glob.glob("stage_05/data/*.json"):
        if os.path.basename(p) in SKIP: continue
        yield json.load(open(p))

checks = {
    "row_count_matches_arms":
        len(rows) == sum(len(r.get("animal_arms") or []) for r in _extracted_records()),
    "required_columns_present": REQUIRED_COLUMNS <= columns,
    "no_row_errors":            not errors,
}
print("Script checks:")
for k, v in checks.items():
    print(f"  {'OK  ' if v else 'FAIL'}  {k}")
if errors:
    print("Errors:")
    for e in errors: print(f"    {e}")

crosstab = Counter((r["species"], r["study_type"]) for r in rows)
n_with_nam = sum(1 for r in rows if (r.get("concurrent_nam") or "").strip())
print(f"\nRows: {len(rows)}  |  with concurrent_nam: {n_with_nam}")
print("species × study_type:")
for (sp, st), n in sorted(crosstab.items()):
    print(f"  {sp:12s} × {st:18s}  {n}")

with open("stage_06/eval/eval_script.json", "w") as f:
    json.dump({
        "script": checks,
        "n_rows": len(rows),
        "n_with_concurrent_nam": n_with_nam,
        "crosstab": [{"species": sp, "study_type": st, "n": n}
                     for (sp, st), n in crosstab.items()],
        "errors": errors,
    }, f, indent=2)

n_pass = sum(1 for v in checks.values() if v); n_total = len(checks)
score_path = "stage_06/eval/score.json"
score = json.load(open(score_path)) if os.path.exists(score_path) else {}
score["script"] = {"passed": n_pass, "total": n_total,
                   "percent": round(100*n_pass/n_total, 1)}
with open(score_path, "w") as f: json.dump(score, f, indent=2)
print(f"\nScore: {n_pass}/{n_total} ({score['script']['percent']}%)")


## 8. Stuck? Skip this stage


Copy the reference run's Stage 6 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil
os.makedirs("stage_06/data", exist_ok=True)
shutil.copy("reference_outputs/stage_06/data/mabs_animal_studies.csv",
            "stage_06/data/mabs_animal_studies.csv")
print("copied reference Stage 6 output")
